# 04 — Simulazione "OASIS-inspired" (Blocco C)

**A cosa serve.** Cruscotto del Blocco C: il lavoro pesante (scenari coi modelli grandi) gira su **HPC**; qui si
1. prova il meccanismo in **locale** con un modello finto (mock);
2. si **caricano** i risultati calcolati su HPC;
3. si **confrontano i modelli** sullo stesso scenario.

I risultati sono organizzati per **esperimento**: `data/processed/simulazioni/<test>/<scenario>/<modello>/`
- `test1_baseline` → 1º test (modelli di taglie diverse, 5 round)
- `test2_stessa_fascia_round10` → 2º test (modelli ~stessa taglia, 10 round)

Ogni run produce `risultato.json` (traccia completa) e `mappa.html` (mappa interattiva).

In [ ]:
import sys
from pathlib import Path
RADICE = Path.cwd().parent
sys.path.insert(0, str(RADICE))

import pandas as pd
from IPython.display import IFrame, display
from src.simulation.scenari import SCENARI, elenco
from src.simulation.run_simulazione import (
    carica, riepilogo, scenari_disponibili, modelli_disponibili, test_disponibili)

print("Scenari definiti:", elenco())
print("Esperimenti disponibili:", test_disponibili())

## 1) Prova locale (mock, senza HPC)
Reazioni **finte** e deterministiche: serve solo a vedere il meccanismo. I modelli veri girano su HPC.

In [ ]:
from src.simulation.oasis_inspired import simula, responder_mock
from src.simulation.mappa_sim import genera

res = simula(SCENARI["siccita_darfur"], responder_mock, n_round=4, verbose=True)
genera(res, titolo="demo_locale",
       out_path=RADICE/"data/processed/graphs/simulazioni/demo_locale.html")
IFrame(src="../data/processed/graphs/simulazioni/demo_locale.html", width="100%", height=560)

## 2) Risultati calcolati su HPC
Scegli l'esperimento (`TEST`), lo scenario e il modello, e guarda la mappa.

In [ ]:
TEST = (test_disponibili() or ["test1_baseline"])[-1]   # <-- l'ultimo esperimento; cambialo a mano se vuoi
SCEN = "siccita_darfur"                                   # <-- scegli lo scenario
print("esperimento:", TEST)
print("scenari disponibili:", scenari_disponibili(TEST))
print(f"modelli per '{SCEN}':", modelli_disponibili(TEST, SCEN))

MOD = (modelli_disponibili(TEST, SCEN) or ["mock"])[0]   # <-- scegli il modello
print("mostro:", TEST, "/", SCEN, "/", MOD)
IFrame(src=f"../data/processed/simulazioni/{TEST}/{SCEN}/{MOD}/mappa.html", width="100%", height=560)

## 3) Confronto tra modelli
Sullo stesso esperimento e scenario: **tabella di sintesi** (round, Paesi reagiti, archi creati/tagliati/
rafforzati/indeboliti, cambi di stato) + le **mappe** dei modelli, per trovarne le differenze.

In [ ]:
TEST = (test_disponibili() or ["test1_baseline"])[-1]   # <-- esperimento da confrontare
SCEN = "siccita_darfur"                                   # <-- scenario da confrontare
mods = modelli_disponibili(TEST, SCEN)

righe = [{"modello": m, **riepilogo(carica(TEST, SCEN, m))} for m in mods if carica(TEST, SCEN, m)]
tab = pd.DataFrame(righe).set_index("modello") if righe else pd.DataFrame()
display(tab)

for m in mods:
    print(f"=== {TEST} · {SCEN} · {m} ===")
    display(IFrame(src=f"../data/processed/simulazioni/{TEST}/{SCEN}/{m}/mappa.html", width="100%", height=520))